# Teach Jumper to jump *toward* something — Kaggle version
Same job as the Colab notebook (fork `jumper.ref_free_jump_dir`, warm-started from Jumper's vertical-jump policy). **Untested on a GPU.**

**Before running:** right panel → *Session options* → **Accelerator: GPU T4 x2 (or P100)** and **Internet: On** (needs a phone-verified account). Then **Save Version → Save & Run All** runs it in the background (up to ~12 h) and keeps `/kaggle/working` as the output. Or press Run on each cell. Cell 6 is a 30-iteration smoke test; if it fails, paste the error.

In [ ]:
# 1. GPU check
!nvidia-smi -L
!nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv

In [ ]:
# 2. Jumper pinned (third-party, Apache-2.0) + this repo
import os, sys, subprocess
W='/kaggle/working'; JUMPER=f'{W}/jumper'; SB=f'{W}/spiderbrain-moveset'
REV='78966f45de6f2b6de3f7cea017f32a39f8b993e6'
if not os.path.exists(JUMPER):
    !git clone https://github.com/KingKongRobotics/jumper.git {JUMPER}
!cd {JUMPER} && git checkout -q {REV} && git log -1 --format='%h %s'
if not os.path.exists(SB):
    !git clone https://github.com/adhritpradeepashoka/spiderbrain-moveset.git {SB}
!cd {SB} && git pull -q && git log -1 --format='%h %s'

In [ ]:
# 3. venv with uv + torch + Jumper (cu126 unless the GPU is very new; P100/T4 -> cu126)
cap = float(subprocess.check_output(['nvidia-smi','--query-gpu=compute_cap','--format=csv,noheader']).decode().split()[0])
CUDA = 'cu128' if cap >= 8.9 else 'cu126'
print('compute cap', cap, '->', CUDA)
PY = f'{JUMPER}/.venv/bin/python'
!pip install -q uv
!rm -rf {JUMPER}/.venv
!uv venv --python {sys.executable} {JUMPER}/.venv
!uv pip install --python {PY} torch==2.9.1 torchvision==0.24.1 --index-url https://download.pytorch.org/whl/{CUDA} 2>&1 | tail -3
!cd {JUMPER} && uv pip install --python {PY} -e . 2>&1 | tail -3
!{PY} -c "import torch; print(torch.cuda.is_available(), torch.cuda.get_device_name(0))"
!{PY} -c "import warp as wp; wp.init(); print(wp.get_devices())" 

In [ ]:
# 4. install the fork task, build the warm-start checkpoint, check it is registered
!{PY} {SB}/jumper_fork/install.py {JUMPER}
!{PY} {SB}/jumper_fork/make_warmstart.py {JUMPER}/tasks/jumper/ref_free_jump/out/example/model_5999.pt {W}/warm.pt
!cd {JUMPER} && {PY} scripts/train.py --list 2>&1 | grep -i "ref_free"
!cd {JUMPER} && {PY} scripts/train.py --task jumper.ref_free_jump_dir --headless --no-tensorboard --dry-run 2>&1 | tail -3

### 5. Knobs
`JUMP_TARGET_MAX` (default 0.30 m) = farthest forward target, ramped in over iterations 100–1500 (`JUMP_RAMP_ITERS=100,1500`). Lower it if the first run can't reach it.

In [ ]:
# 6. SMOKE TEST: 30 iterations, 512 envs (use 256 if you see an out-of-memory error)
!cd {JUMPER} && JUMP_RAMP_ITERS=0,20 {PY} scripts/train.py --task jumper.ref_free_jump_dir --checkpoint {W}/warm.pt --headless --no-tensorboard --num_envs 512 --max-iterations 30 2>&1 | tail -60

In [ ]:
# 7. LONG RUN (foreground; 1500 iterations first -- bump to 3000 if it is learning). Checkpoints land in jumper/logs and are copied to /kaggle/working/ckpts.
os.environ.update(JUMP_TARGET_MAX='0.30', JUMP_RAMP_ITERS='100,1500', MUJOCO_GL='egl')
!cd {JUMPER} && {PY} scripts/train.py --task jumper.ref_free_jump_dir --checkpoint {W}/warm.pt --headless --no-tensorboard --max-iterations 1500 > {W}/train.log 2>&1
!grep -E "landing_err|target_fwd|Mean reward|Traceback|Error" {W}/train.log | tail -15
!mkdir -p {W}/ckpts && find {JUMPER}/logs -name "model_*.pt" -newer {W}/warm.pt | sort | tail -3 | xargs -I{} cp {} {W}/ckpts/

In [ ]:
# 8. RENDER the newest checkpoint with a fixed target (edit JUMP_TARGET_X/Y), side camera
os.environ.update(JUMP_TARGET_X='0.20', JUMP_TARGET_Y='0.0', MUJOCO_GL='egl')
!cd {JUMPER} && {PY} scripts/play.py --task jumper.ref_free_jump_dir --headless --steps 600 --video {W}/dir_jump.mp4 --no-video-follow --video-lookat-height 0.08 --video-distance 1.6 --video-azimuth 90 2>&1 | tail -15
from IPython.display import Video
Video(f'{W}/dir_jump.mp4', embed=True, width=640)

Outputs (`train.log`, `ckpts/`, `dir_jump.mp4`) appear under the notebook's **Output** tab after the run; download from there.